![Logo ADL](https://github.com/nicolastibata/MINE_4210_ADL_202520/blob/main/docs/images/logo.png?raw=true)


# **Laboratorio 9: BERT + Generación de texto**
**Tutor: Nicolás Tibatá**

## **Tabla de Contenido**

[Contexto y objetivos](#scrollTo=5KnQpgpopi8a)<br>
[1. Introducción de los datos](#scrollTo=VjA8zwzJvmeO)<br>
[2. Preparación y Modelamiento](#scrollTo=kG8XHROzvuEH)<br>
[3. Taller 4](#scrollTo=JTKc52_Wvs_N)<br>

### **Contexto y Objetivos**
## Contexto y Objetivos

- Se requiere realizar el análisis de sentimientos de un conjunto de frases del sector financiero como parte de la evaluación del sentimiento del mercado y la reputación de empresas, con el objetivo de tomar decisiones de inversión informadas.

### **Objetivos**
1. Construir una Red Neuronal basada en una arquitectura Transformers para llevar a cabo un análisis de sentimientos, ejemplificando así la aplicación de modelos de procesamiento del lenguaje natural.
2. Generar texto en base a gpt2 de manera de ejemplo.
3. Aplicar de manera combinada ambas herramientas en el taller 4.



**Datos:** [finance-sentence](https://www.kaggle.com/datasets/sbhatti/financial-sentiment-analysis)

### **1. Introducción a los datos**

In [45]:
!pip install torch -q
!pip install keras-tuner -q
!pip install transformers -q
!pip install "tf-models-official==2.13.*" -q

  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [46]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
import shutil
import pandas as pd
import numpy as np
import seaborn as sns


import matplotlib.pyplot as plt
%matplotlib inline

import tensorflow as tf
import keras_tuner as kt

from keras.models import Sequential
from keras.layers import Input, Dense, Dropout, LSTM, Flatten
from transformers import TFBertForSequenceClassification, BertTokenizer

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
from google.colab import files

import tensorflow_hub as hub
import tensorflow_text

from google.colab import files
from google.colab import userdata

In [47]:
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')

!kaggle datasets download -d sbhatti/financial-sentiment-analysis
!unzip "financial-sentiment-analysis.zip"

Dataset URL: https://www.kaggle.com/datasets/sbhatti/financial-sentiment-analysis
License(s): CC0-1.0
financial-sentiment-analysis.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  financial-sentiment-analysis.zip
replace data.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: data.csv                


In [4]:
# Devidir los datos en train y test
data = pd.read_csv('/content/data.csv', sep=',')
data

,Sentence,Sentiment
0,The GeoSolutions technology will leverage Bene...,positive
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,According to the Finnish-Russian Chamber of Co...,neutral
4,The Swedish buyout firm has sold its remaining...,neutral
...,...,...
5837,RISING costs have forced packaging producer Hu...,negative
5838,Nordic Walking was first used as a summer trai...,neutral
5839,"According shipping company Viking Line , the E...",neutral
5840,"In the building and home improvement trade , s...",neutral


Veamos la distribución de los sentimientos en el dataset

In [5]:
data["Sentiment"].value_counts()

,count
Sentiment,
neutral,3130
positive,1852
negative,860


### **2. Preparación y Modelamiento**

#### **Preparación**

Pasamos el sentimiento (clase) a un valor entero usando LabelEncoder

In [6]:
label_encoder = LabelEncoder()
data['Sentiment'] = label_encoder.fit_transform(data['Sentiment'])

etiquetas_unicas = label_encoder.classes_
for valor_numerico, etiqueta_original in enumerate(etiquetas_unicas):
    print(f'Valor numérico: {valor_numerico}, Etiqueta original: {etiqueta_original}')

Valor numérico: 0, Etiqueta original: negative
Valor numérico: 1, Etiqueta original: neutral
Valor numérico: 2, Etiqueta original: positive


In [7]:
# Divide los datos en entrenamiento y prueba
train, test = train_test_split(data, test_size=0.2, stratify=data['Sentiment'], random_state=42, shuffle = True)

# Ahora divide el conjunto de entrenamiento en entrenamiento y validación
train, val = train_test_split(train, test_size=0.2, stratify=train['Sentiment'], random_state=42, shuffle = True)

print("Tamaño de datos de entrenamiento:", train.shape)
print("Tamaño de datos de validación:", val.shape)
print("Tamaño de datos de prueba:", test.shape)

train

Tamaño de datos de entrenamiento: (3738, 2)
Tamaño de datos de validación: (935, 2)
Tamaño de datos de prueba: (1169, 2)


,Sentence,Sentiment
3097,"Digia will also set up two subsidiaries , Digi...",1
5048,$BBRY Sierra. Has a great cash balance and imp...,2
5727,"Britain's FTSE gains, Land Securities up after...",2
185,The Finnish company sold its UK operation - co...,1
4265,Russian Media Ventures ' minority shareholder ...,1
...,...,...
326,( I&H ) in a move to enhance growth .,2
2821,"In addition , a further 29 employees can be la...",0
4365,"The paper industry 's de-inking sludge , which...",1
1603,$JE LOOKS like we are bouncing. Would be nice...,2


In [8]:
X_train, X_test, X_val= train['Sentence'], test['Sentence'], val['Sentence']
y_train, y_test, y_val= train['Sentiment'], test['Sentiment'], val['Sentiment']

print("x_train", X_train.shape, " y_train:", y_train.shape)
print("x_test:", X_test.shape, "y_test", y_test.shape)
print("x_val", X_val.shape, "y_val:", y_val.shape)

x_train (3738,)  y_train: (3738,)
x_test: (1169,) y_test (1169,)
x_val (935,) y_val: (935,)


#### **Modelamiento BERT**

 Con el código de las siguientes secciones podemos seleccionar uno de los modelos BERT y su correspondiente modelo de preprocesamiento utilizando TensorFlow Hub. Pueden ver más información [aquí](https://huggingface.co/transformers/v3.0.2/model_doc/bert.html).

In [9]:
# Cargar los modelos de TensorFlow Hub con trainable=True si deseas fine-tuning
bert_preprocess_model = hub.KerasLayer(
    "https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3",
    name='preprocessing'
)

bert_model = hub.KerasLayer(
    "https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-4_H-512_A-8/2",
    trainable=False, # Solo como extractor de características
    name='BERT_encoder'
)

print(f'BERT model selected: {bert_model}')
print(f'Preprocess model selected: {bert_preprocess_model}')

BERT model selected: <tensorflow_hub.keras_layer.KerasLayer object at 0x7d2834125580>
Preprocess model selected: <tensorflow_hub.keras_layer.KerasLayer object at 0x7d28341257c0>


In [10]:
# Capas Bert
text_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='text')
preprocessing_output = bert_preprocess_model(text_input)
outputs = bert_model(preprocessing_output)

# Capas de nuestra red adicional
#l = tf.keras.layers.Dropout(0.1, name="dropout")
l = tf.keras.layers.Dense(512, activation='sigmoid', name="capaOculta1")(outputs['pooled_output'])
l = tf.keras.layers.Dropout(0.2, name="dropout")(l)
y = tf.keras.layers.Dense(3, activation='softmax', name="output")(l)

# Usamos inputs y outputs para construir el modelo final
model = tf.keras.Model(inputs=[text_input], outputs = [y])

In [11]:
metrics = ['accuracy']

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=metrics)

In [12]:
print(model.summary())

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text (InputLayer)           [(None,)]                    0         []                            
                                                                                                  
 preprocessing (KerasLayer)  {'input_type_ids': (None,    0         ['text[0][0]']                
                             128),                                                                
                              'input_mask': (None, 128)                                           
                             , 'input_word_ids': (None,                                           
                              128)}                                                               
                                                                                              

In [13]:
model.fit(X_train, y_train,
          validation_data = (X_val, y_val),
          epochs=20,
          #callbacks= EarlyStopping(monitor='val_accuracy', patience=4)
          )

Epoch 1/20
117/117 [==============================] - 19s 100ms/step - loss: 0.8865 - accuracy: 0.5947 - val_loss: 0.7554 - val_accuracy: 0.6460
Epoch 2/20
117/117 [==============================] - 10s 90ms/step - loss: 0.7672 - accuracy: 0.6479 - val_loss: 0.7197 - val_accuracy: 0.6727
Epoch 3/20
117/117 [==============================] - 11s 90ms/step - loss: 0.7486 - accuracy: 0.6584 - val_loss: 0.7246 - val_accuracy: 0.6310
Epoch 4/20
117/117 [==============================] - 10s 90ms/step - loss: 0.7161 - accuracy: 0.6707 - val_loss: 0.6885 - val_accuracy: 0.6834
Epoch 5/20
117/117 [==============================] - 10s 90ms/step - loss: 0.7066 - accuracy: 0.6827 - val_loss: 0.7109 - val_accuracy: 0.6642
Epoch 6/20
117/117 [==============================] - 10s 89ms/step - loss: 0.6896 - accuracy: 0.6822 - val_loss: 0.7005 - val_accuracy: 0.6492
Epoch 7/20
117/117 [==============================] - 10s 87ms/step - loss: 0.6786 - accuracy: 0.6822 - val_loss: 0.6713 - val_accuracy

**Evaluamos nuestro modelo**

In [14]:
y_pred = model.predict(X_train)
y_pred = np.argmax(y_pred, axis=1)

print(classification_report(y_train, y_pred))

117/117 [==============================] - 9s 70ms/step
              precision    recall  f1-score   support

           0       0.52      0.63      0.57       550
           1       0.82      0.84      0.83      2003
           2       0.79      0.68      0.73      1185

    accuracy                           0.76      3738
   macro avg       0.71      0.72      0.71      3738
weighted avg       0.77      0.76      0.76      3738



In [15]:
y_pred = model.predict(X_test)
y_pred = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred))

37/37 [==============================] - 2s 67ms/step
              precision    recall  f1-score   support

           0       0.36      0.40      0.38       172
           1       0.78      0.82      0.80       626
           2       0.70      0.60      0.65       371

    accuracy                           0.69      1169
   macro avg       0.61      0.61      0.61      1169
weighted avg       0.69      0.69      0.69      1169



#### **Generación de Texto**

Utilizamos un modelo GPT transformer para generación de texto, el proceso es similar al uso de BERT

In [16]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

In [17]:
model_name = "gpt2"
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [18]:
input_text = "Once upon a time"
input_ids = tokenizer.encode(input_text, return_tensors="pt")

In [19]:
output = model.generate(
    input_ids,
    max_length=50,           # Longitud máxima del texto generado
    num_return_sequences=1,  # Cantidad de sentencias generadas
    temperature=0.7,         # Temperatura (más alto = más creativo)
    top_k=50,                # Top 50 de palabras probables
    top_p=0.9,               # Centrado a palabras de alta probabilidad
    repetition_penalty=1.2,  # Reducción de repetición de palabras
    do_sample=True           # Habilita el randomness del texto generado
)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [20]:
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)

Once upon a time, the city was besieged by an army of mercenaries. The men had been sent to kill them and they were all killed as well—the guardsman for whom we are speaking died during this war."
The king then asked us


### **3. Taller 4**


Instrucciones

1. El archivo a presentar debe ser en formato .ipynb o HTML con sus celdas ejecutadas. Celdas sin ejecutar no podrán ser evaluadas.
2. El nombre del archivo debe ser taller_4_{Apellido_Nombre}_{Apellido_Nombre} de cada integrante del equipo.
3. Las entregas solo se hacen a través de Bloque Neón.

------


1. Los resultados del modelo base no son los mejores. Utilice el modelo [Distilbert](https://huggingface.co/transformers/v3.0.2/model_doc/distilbert.html) de Huggingface aplicando padding a los diferentes dataframes (train, val, test). ¿Qué es Distilbert? ¿Mejoran los resultados? ¿El entrenamiento es más rápido? Compare los resultados con el modelo base del laboratorio.
```python
# clue

import tensorflow as tf
from transformers import DistilBertTokenizer, TFDistilBertForSequenceClassification
from sklearn.model_selection import train_test_split
# Tokenización
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
X_train = [tokenizer.encode(text, add_special_tokens=True, truncation=True, max_length=128) for text in X_train]
X_val = [tokenizer.encode(text, add_special_tokens=True, truncation=True, max_length=128) for text in X_val]
X_test = [tokenizer.encode(text, add_special_tokens=True, truncation=True, max_length=128) for text in X_test]

# Padding
X_train = tf.keras.preprocessing.sequence.pad_sequences(...)
X_val = tf.keras.preprocessing.sequence.pad_sequences(...)
X_test = tf.keras.preprocessing.sequence.pad_sequences(...)

model = TFDistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased")
```

2. Genere sentencias nuevas con el modelo gpt2 o algún otro de preferencia para poder tener datos sintéticos y aumentar la cantidad de sentencias negativas del dataset original (aumentar las sentencias negativas hasta 1200 datos de esa clase). Luego reentrene el modelo del punto 1 (es decir el modelo entrenado con Distilbert) con este nuevo dataset aumentado y documente las métricas de evaluación.


3. Realice fine-tuning al modelo base del laboratorio y compárelo con el modelo del punto 2. ¿Que diferencias nota al realizar este paso adicional?
``` python
# clue

trainable=True,

```


**Desarrollo del Taller**

**Punto 1**

A continuación se incluyen las celdas con el código necesario para utilizar Distilbert, se hace una primera configuración para utilizar DISTILBERT con TensorFlow sin embargo, no fué posible la ejecución del modelo por incompatibilidad de clases, la plataforma recomienda la utilización de Clases PyTorch.


In [ ]:
# clue

import tensorflow as tf
from transformers import DistilBertTokenizer, TFDistilBertForSequenceClassification
from sklearn.model_selection import train_test_split

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# Reemplaza todo tu código de tokenización por esto:
X_train = [tokenizer.encode(text, add_special_tokens=True, truncation=True, max_length=128) for text in X_train]
X_val = [tokenizer.encode(text, add_special_tokens=True, truncation=True, max_length=128) for text in X_val]
X_test = [tokenizer.encode(text, add_special_tokens=True, truncation=True, max_length=128) for text in X_test]

from tensorflow.keras.preprocessing.sequence import pad_sequences

# Aplicar padding después de tokenizar
X_train = pad_sequences(X_train, maxlen=128, dtype='int32', padding='post', truncating='post')
X_val = pad_sequences(X_val, maxlen=128, dtype='int32', padding='post', truncating='post')
X_test = pad_sequences(X_test, maxlen=128, dtype='int32', padding='post', truncating='post')

model = TFDistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3, local_files_only=False)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


TypeError: 'builtins.safe_open' object is not iterable

Uso del modelo Distilbert de Huggingface con Pytorch y agregando la capa de personalización equivalente utilizada previamente con BERT

******************************************************
**Iniciamos con la carga de Distilbert congelando este modelo base**
*****************************************************

In [26]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm
import numpy as np

# 1. Verificar si hay GPU disponible
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando dispositivo: {device}')

# 2. Cargar el tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# 3. Tokenizar y aplicar padding a los dataframes
X_train = tokenizer(
    train['Sentence'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'  # PyTorch tensors
)

X_val = tokenizer(
    val['Sentence'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

X_test = tokenizer(
    test['Sentence'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

# 4. Crear clase Dataset personalizada
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item

# 5. Crear datasets
train_dataset = SentimentDataset(X_train, train['Sentiment'])
val_dataset = SentimentDataset(X_val, val['Sentiment'])
test_dataset = SentimentDataset(X_test, test['Sentiment'])

# 6. Crear DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)



Usando dispositivo: cuda


**Continuamos con la personalizacion del modelo para incluir 3 capas:**

*   Capa densa oculta de 512 neuronas y función de activacion sigmoide
*   Dropout de 20%
*   Capa densa de salida con función softmax

In [27]:
# 7. Crear modelo personalizado con capas adicionales
class DistilBertWithCustomLayers(torch.nn.Module):
    def __init__(self, num_labels=3):
        super(DistilBertWithCustomLayers, self).__init__()

        # Cargar DistilBERT base
        self.distilbert = DistilBertForSequenceClassification.from_pretrained(
            'distilbert-base-uncased',
            num_labels=num_labels
        ).distilbert  # Solo el encoder, sin la capa de clasificación

        # Capas personalizadas
        self.dense_hidden = torch.nn.Linear(768, 512)  # 768 (DistilBERT) → 512 neuronas
        self.sigmoid = torch.nn.Sigmoid()              # Activación sigmoide
        self.dropout = torch.nn.Dropout(0.2)           # Dropout 20%
        self.classifier = torch.nn.Linear(512, num_labels)  # 512 → 3 neuronas
        self.softmax = torch.nn.Softmax(dim=1)         # Activación softmax

    def forward(self, input_ids, attention_mask):
        # Pasar por DistilBERT
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Obtener el embedding del token [CLS] (primer token)
        hidden_state = outputs.last_hidden_state[:, 0, :]  # [batch_size, 768]

        # Pasar por capas personalizadas
        x = self.dense_hidden(hidden_state)  # [batch_size, 512] - Capa densa 512
        x = self.sigmoid(x)                  # Activación sigmoide
        x = self.dropout(x)                  # Dropout 0.2
        x = self.classifier(x)               # [batch_size, 3] - Capa de salida
        logits = self.softmax(x)             # Activación softmax

        return logits

# Instanciar el modelo
model = DistilBertWithCustomLayers(num_labels=3)

# ============ CONGELAR DISTILBERT ============
for param in model.distilbert.parameters():
    param.requires_grad = False
# ============================================

# Contar parámetros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parámetros: {total_params:,}")
print(f"Parámetros entrenables: {trainable_params:,}")
print(f"Parámetros congelados: {total_params - trainable_params:,}")

model.to(device)

# ==================== VISUALIZACIÓN DE ARQUITECTURA ====================
print("\n" + "="*80)
print("ARQUITECTURA DEL MODELO - DistilBERT + Capas Personalizadas")
print("="*80)
print(model)
print("-"*80)
print(f"\n{'RESUMEN':^80}")
print("-"*80)
print(f"Total de parámetros:      {total_params:>20,}")
print(f"Parámetros entrenables:   {trainable_params:>20,}")
print(f"Parámetros congelados:    {total_params - trainable_params:>20,}")
print(f"Porcentaje entrenable:    {100 * trainable_params / total_params:>19.2f}%")
print("="*80 + "\n")
# ========================================================================



Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Total parámetros: 66,758,147
Parámetros entrenables: 395,267
Parámetros congelados: 66,362,880

ARQUITECTURA DEL MODELO - DistilBERT + Capas Personalizadas
DistilBertWithCustomLayers(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
        

**Entrenamiento del nuevo modelo:**

In [28]:
# 8. Configurar optimizador y scheduler
optimizer = AdamW(model.parameters(), lr=5e-5)
num_epochs = 20
total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# 9. Función de entrenamiento (MODIFICADA)
def train_epoch(model, data_loader, optimizer, scheduler, device):
    model.train()
    losses = []
    correct_predictions = 0
    loss_fn = torch.nn.NLLLoss()  # ⬅️ CAMBIO: NLLLoss para usar con softmax

    for batch in tqdm(data_loader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Aplicar log antes de calcular la pérdida
        log_probs = torch.log(logits + 1e-10)  # Añadir epsilon para estabilidad numérica
        loss = loss_fn(log_probs, labels)

        # Calcular accuracy
        _, preds = torch.max(logits, dim=1)
        correct_predictions += torch.sum(preds == labels)

        losses.append(loss.item())

        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    return correct_predictions.double() / len(data_loader.dataset), np.mean(losses)


# 10. Función de evaluación (MODIFICADA)
def eval_model(model, data_loader, device):
    model.eval()
    losses = []
    correct_predictions = 0
    loss_fn = torch.nn.NLLLoss()  # ⬅️ CAMBIO: NLLLoss para usar con softmax

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            # Aplicar log antes de calcular la pérdida
            log_probs = torch.log(logits + 1e-10)  # Añadir epsilon para estabilidad numérica
            loss = loss_fn(log_probs, labels)

            _, preds = torch.max(logits, dim=1)
            correct_predictions += torch.sum(preds == labels)
            losses.append(loss.item())

    return correct_predictions.double() / len(data_loader.dataset), np.mean(losses)

# 11. Entrenamiento
history = {'train_acc': [], 'train_loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    print('-' * 50)

    train_acc, train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    print(f'Train loss: {train_loss:.4f}, Train accuracy: {train_acc:.4f}')

    val_acc, val_loss = eval_model(model, val_loader, device)
    print(f'Val loss: {val_loss:.4f}, Val accuracy: {val_acc:.4f}')

    history['train_acc'].append(train_acc)
    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc)
    history['val_loss'].append(val_loss)


Epoch 1/20
--------------------------------------------------


Training: 100%|██████████| 117/117 [00:11<00:00, 10.15it/s]


Train loss: 0.9955, Train accuracy: 0.5040


Evaluating: 100%|██████████| 15/15 [00:02<00:00,  5.14it/s]


Val loss: 0.9391, Val accuracy: 0.5358

Epoch 2/20
--------------------------------------------------


...

Val loss: 0.7578, Val accuracy: 0.6321

Epoch 20/20
--------------------------------------------------


Training: 100%|██████████| 117/117 [00:11<00:00, 10.13it/s]


Train loss: 0.7645, Train accuracy: 0.6568


Evaluating: 100%|██████████| 15/15 [00:02<00:00,  5.19it/s]

Val loss: 0.7573, Val accuracy: 0.6332


**Evaluación final del nuevo modelo:**

In [30]:
# 12. Evaluación final en training
test_acc, test_loss = eval_model(model, train_loader, device)
print(f'\nTrain loss: {test_loss:.4f}, Train accuracy: {test_acc:.4f}')

print('\n' * 2)

# 12. Evaluación final en test
test_acc, test_loss = eval_model(model, test_loader, device)
print(f'\nTest loss: {test_loss:.4f}, Test accuracy: {test_acc:.4f}')

Evaluating: 100%|██████████| 117/117 [00:11<00:00, 10.33it/s]



Train loss: 0.7563, Train accuracy: 0.6514





Evaluating: 100%|██████████| 19/19 [00:03<00:00,  5.15it/s]


Test loss: 0.7531, Test accuracy: 0.6638


**Punto 2 - Generación de datos con gpt2**

Se agregan celdas de código para utilizar Generador GPT2 y generar de manera sintética 340 sentencias negativas que se agregaran al dataset inicial para incrementar la cantidad de datos de la clase de Sentimiento negativo a 1200

In [31]:
from transformers import pipeline
import pandas as pd
import numpy as np

print("🔄 Cargando modelo GPT-2...")
# Crear pipeline de generación de texto (usa PyTorch en el backend)
generator = pipeline('text-generation', model='gpt2', framework='pt')

# Prompts para generar sentencias negativas
negative_prompts = [
    "I hate",
    "I'm disappointed",
    "This is terrible",
    "I feel sad",
    "The worst thing",
    "I'm angry about",
    "I dislike",
    "This makes me upset",
    "I regret",
    "I'm frustrated by",
    "It's awful",
    "I can't stand",
    "This is horrible",
    "I feel terrible",
    "What a disaster",
    "I'm unhappy with",
    "This is disappointing",
    "I'm annoyed by",
    "I despise",
    "This ruins",
    "I'm sick of",
    "This bothers me",
    "I'm worried about",
    "This is unacceptable",
    "I feel miserable",
    "This is the worst",
    "I'm depressed about",
    "This makes me furious",
    "I'm disgusted by",
    "This is so bad",
    "I never want to",
    "This hurts",
    "I'm upset that",
    "This is painful",
]

sentences = []
num_sentences = 340
sentences_per_prompt = num_sentences // len(negative_prompts) + 1

print(f"\n📝 Generando {num_sentences} sentencias con sentimiento negativo...\n")

for i, prompt in enumerate(negative_prompts):
    if len(sentences) >= num_sentences:
        break

    # Calcular cuántas sentencias generar con este prompt
    needed = min(sentences_per_prompt, num_sentences - len(sentences))

    for j in range(needed):
        try:
            # Generar texto
            result = generator(
                prompt,
                max_new_tokens=20,       # Longitud máxima de nuevos tokens generados
                num_return_sequences=1,  # Una sentencia a la vez
                temperature=0.9,         # Creatividad
                top_k=50,
                top_p=0.95,
                do_sample=True,
                pad_token_id=50256,      # Token de padding de GPT-2
                truncation=True
            )

            # Extraer y limpiar la sentencia
            sentence = result[0]['generated_text'].strip()

            # Tomar solo la primera oración (hasta el primer punto)
            if '.' in sentence:
                sentence = sentence.split('.')[0] + '.'
            elif '!' in sentence:
                sentence = sentence.split('!')[0] + '!'
            elif '?' in sentence:
                sentence = sentence.split('?')[0] + '?'
            else:
                sentence = sentence + '.'

            # Limpiar saltos de línea y espacios múltiples
            sentence = ' '.join(sentence.split())

            sentences.append(sentence)

            # Mostrar progreso cada 50 sentencias
            if len(sentences) % 50 == 0:
                print(f"✓ Generadas {len(sentences)}/{num_sentences} sentencias...")

        except Exception as e:
            print(f"⚠️ Error generando sentencia: {e}")
            continue

# Asegurar que tenemos exactamente 340 sentencias
sentences = sentences[:num_sentences]

print(f"\n✅ Se generaron {len(sentences)} sentencias con éxito!\n")

# Crear DataFrame
df = pd.DataFrame({
    'Sentence': sentences,
    'Sentiment': ['negative'] * len(sentences)  # Todas etiquetadas como negativas
})

# Mostrar primeras 10 sentencias
print("="*80)
print("PRIMERAS 10 SENTENCIAS GENERADAS:")
print("="*80)
for idx, row in df.head(10).iterrows():
    print(f"{idx+1}. {row['Sentence']}")
    print(f"   Sentimiento: {row['Sentiment']}\n")

# Mostrar estadísticas
print("="*80)
print("ESTADÍSTICAS:")
print("="*80)
print(f"Total de sentencias:        {len(df)}")
print(f"Sentencias negativas:       {(df['Sentiment'] == 'negative').sum()}")
print(f"Longitud promedio:          {df['Sentence'].str.len().mean():.1f} caracteres")
print(f"Longitud mínima:            {df['Sentence'].str.len().min()} caracteres")
print(f"Longitud máxima:            {df['Sentence'].str.len().max()} caracteres")
print("="*80)

# Guardar en CSV
csv_filename = 'negative_sentences_340.csv'
df.to_csv(csv_filename, index=False, encoding='utf-8')
print(f"\n💾 Archivo guardado: '{csv_filename}'")

# Guardar también en formato texto plano
txt_filename = 'negative_sentences_340.txt'
with open(txt_filename, 'w', encoding='utf-8') as f:
    for idx, row in df.iterrows():
        f.write(f"{idx+1}. {row['Sentence']} | Sentimiento: {row['Sentiment']}\n")
print(f"💾 Archivo guardado: '{txt_filename}'")

# Mostrar algunas sentencias aleatorias
print("\n" + "="*80)
print("10 SENTENCIAS ALEATORIAS:")
print("="*80)
random_samples = df.sample(n=10, random_state=42)
for idx, row in random_samples.iterrows():
    print(f"• {row['Sentence']}")

print("\n✨ ¡Proceso completado exitosamente!")

🔄 Cargando modelo GPT-2...


Device set to use cuda:0



📝 Generando 340 sentencias con sentimiento negativo...



You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


✓ Generadas 50/340 sentencias...
✓ Generadas 100/340 sentencias...
✓ Generadas 150/340 sentencias...
✓ Generadas 200/340 sentencias...
✓ Generadas 250/340 sentencias...
✓ Generadas 300/340 sentencias...

✅ Se generaron 340 sentencias con éxito!

PRIMERAS 10 SENTENCIAS GENERADAS:
1. I hate how hard it is to go from the low hanging fruit in any country to a full time job.
   Sentimiento: negative

2. I hate to say it, but he is the only person who does not agree with his views on guns.
   Sentimiento: negative

3. I hate to say it, but it's an awful lot of work, but it's an awful lot of.
   Sentimiento: negative

4. I hate it when I go to the airport, and I'm like, 'Oh my God, how does.
   Sentimiento: negative

5. I hate to say this, but I've been through my fair share of things in my life.
   Sentimiento: negative

6. I hate to take your life, it's going to hurt to talk about the way I feel about myself,".
   Sentimiento: negative

7. I hate this game.
   Sentimiento: negative

8. I ha

**Punto 2 - Ampliación del Dataset**

Combinamos los 2 archivos, el de sentimientos original con los datos sintéticos generados con gpt2, se genera un archivo data2.csv

In [33]:
import pandas as pd
import numpy as np

# Cargar datasets
df1 = pd.read_csv('data.csv')
df2 = pd.read_csv('negative_sentences_340.csv')

# Combinar
df_combined = pd.concat([df1, df2], ignore_index=True)

# Mezclar aleatoriamente (importante para el entrenamiento)
df_combined_shuffled = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset 1: {len(df1)} filas")
print(f"Dataset 2: {len(df2)} filas")
print(f"Dataset combinado y mezclado: {len(df_combined_shuffled)} filas")

# Mostrar antes y después de mezclar
print("\nPrimeras 5 filas ANTES de mezclar:")
print(df_combined.head())

print("\nPrimeras 5 filas DESPUÉS de mezclar:")
print(df_combined_shuffled.head())

# Guardar
df_combined_shuffled.to_csv('data2.csv', index=False)
print("\n✓ Dataset mezclado guardado!")


Dataset 1: 5842 filas
Dataset 2: 340 filas
Dataset combinado y mezclado: 6182 filas

Primeras 5 filas ANTES de mezclar:
                                            Sentence Sentiment
0  The GeoSolutions technology will leverage Bene...  positive
1  $ESI on lows, down $1.50 to $2.50 BK a real po...  negative
2  For the last quarter of 2010 , Componenta 's n...  positive
3  According to the Finnish-Russian Chamber of Co...   neutral
4  The Swedish buyout firm has sold its remaining...   neutral

Primeras 5 filas DESPUÉS de mezclar:
                                            Sentence Sentiment
0  Profit for the period was EUR 9.8 mn , up from...  positive
1  Basware Einvoices Oy will be merged into the p...   neutral
2  GyPSii service supports ten different language...   neutral
3  The airline 's share price closed down slightl...  negative
4  Her present position is the director of Stockm...   neutral

✓ Dataset mezclado guardado!


**Punto 2 - Preparación de nuevos datos**

Se regeneran los dataframes tomando el nuevo archivo generado

In [48]:
# Devidir los datos en train y test
data = pd.read_csv('/content/data2.csv', sep=',')
data

,Sentence,Sentiment
0,"Profit for the period was EUR 9.8 mn , up from...",positive
1,Basware Einvoices Oy will be merged into the p...,neutral
2,GyPSii service supports ten different language...,neutral
3,The airline 's share price closed down slightl...,negative
4,Her present position is the director of Stockm...,neutral
...,...,...
6177,@BULLYA @pollux654321 My 50 $KORS 80 Calls are...,positive
6178,"According to Sepp+ñnen , the new technology UM...",positive
6179,$CRUS Upgraded to a buy by Alpha Street Research,positive
6180,Favourable currency rates also contributed to ...,positive


In [49]:
data["Sentiment"].value_counts()

,count
Sentiment,
neutral,3130
positive,1852
negative,1200


In [50]:
label_encoder = LabelEncoder()
data['Sentiment'] = label_encoder.fit_transform(data['Sentiment'])

etiquetas_unicas = label_encoder.classes_
for valor_numerico, etiqueta_original in enumerate(etiquetas_unicas):
    print(f'Valor numérico: {valor_numerico}, Etiqueta original: {etiqueta_original}')

Valor numérico: 0, Etiqueta original: negative
Valor numérico: 1, Etiqueta original: neutral
Valor numérico: 2, Etiqueta original: positive


In [51]:
# Divide los datos en entrenamiento y prueba
train, test = train_test_split(data, test_size=0.2, stratify=data['Sentiment'], random_state=42, shuffle = True)

# Ahora divide el conjunto de entrenamiento en entrenamiento y validación
train, val = train_test_split(train, test_size=0.2, stratify=train['Sentiment'], random_state=42, shuffle = True)

print("Tamaño de datos de entrenamiento:", train.shape)
print("Tamaño de datos de validación:", val.shape)
print("Tamaño de datos de prueba:", test.shape)

train

Tamaño de datos de entrenamiento: (3956, 2)
Tamaño de datos de validación: (989, 2)
Tamaño de datos de prueba: (1237, 2)


,Sentence,Sentiment
3489,Tieto was looking for an energy solution which...,2
5457,Pharmaceuticals group Orion Corp reported a fa...,0
1529,Finnish power supply solutions and systems pro...,1
1225,"TVO 's two-unit 1,740 MW Olkiluoto plant gener...",1
135,"As an alternative to the share exchange , Pano...",1
...,...,...
3970,Exel wants to serve its industrial customers w...,1
4451,"The Lemminkainen Group , headquartered in Hels...",1
3492,CompaniesandMarkets.com provides a wide range ...,1
3910,$BBRY Sierra. Has a great cash balance and imp...,2


In [52]:
X_train, X_test, X_val= train['Sentence'], test['Sentence'], val['Sentence']
y_train, y_test, y_val= train['Sentiment'], test['Sentiment'], val['Sentiment']

print("x_train", X_train.shape, " y_train:", y_train.shape)
print("x_test:", X_test.shape, "y_test", y_test.shape)
print("x_val", X_val.shape, "y_val:", y_val.shape)

x_train (3956,)  y_train: (3956,)
x_test: (1237,) y_test (1237,)
x_val (989,) y_val: (989,)


**Punto - 2 Nueva Carga y Reentrenamiento**

A continuacion se recargan los dataframes para tokenizar y posteriormente reentrenar

In [39]:
# 1. Verificar si hay GPU disponible
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando dispositivo: {device}')

# 2. Cargar el tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# 3. Tokenizar y aplicar padding a los dataframes
X_train = tokenizer(
    train['Sentence'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'  # PyTorch tensors
)

X_val = tokenizer(
    val['Sentence'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

X_test = tokenizer(
    test['Sentence'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

# 4. Crear clase Dataset personalizada
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item

# 5. Crear datasets
train_dataset = SentimentDataset(X_train, train['Sentiment'])
val_dataset = SentimentDataset(X_val, val['Sentiment'])
test_dataset = SentimentDataset(X_test, test['Sentiment'])

# 6. Crear DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)



Usando dispositivo: cuda


In [40]:
# 7. Crear modelo personalizado con capas adicionales
class DistilBertWithCustomLayers(torch.nn.Module):
    def __init__(self, num_labels=3):
        super(DistilBertWithCustomLayers, self).__init__()

        # Cargar DistilBERT base
        self.distilbert = DistilBertForSequenceClassification.from_pretrained(
            'distilbert-base-uncased',
            num_labels=num_labels
        ).distilbert  # Solo el encoder, sin la capa de clasificación

        # Capas personalizadas
        self.dense_hidden = torch.nn.Linear(768, 512)  # 768 (DistilBERT) → 512 neuronas
        self.sigmoid = torch.nn.Sigmoid()              # Activación sigmoide
        self.dropout = torch.nn.Dropout(0.2)           # Dropout 20%
        self.classifier = torch.nn.Linear(512, num_labels)  # 512 → 3 neuronas
        self.softmax = torch.nn.Softmax(dim=1)         # Activación softmax

    def forward(self, input_ids, attention_mask):
        # Pasar por DistilBERT
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Obtener el embedding del token [CLS] (primer token)
        hidden_state = outputs.last_hidden_state[:, 0, :]  # [batch_size, 768]

        # Pasar por capas personalizadas
        x = self.dense_hidden(hidden_state)  # [batch_size, 512] - Capa densa 512
        x = self.sigmoid(x)                  # Activación sigmoide
        x = self.dropout(x)                  # Dropout 0.2
        x = self.classifier(x)               # [batch_size, 3] - Capa de salida
        logits = self.softmax(x)             # Activación softmax

        return logits

# Instanciar el modelo
model = DistilBertWithCustomLayers(num_labels=3)

# ============ CONGELAR DISTILBERT ============
for param in model.distilbert.parameters():
    param.requires_grad = False
# ============================================

# Contar parámetros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parámetros: {total_params:,}")
print(f"Parámetros entrenables: {trainable_params:,}")
print(f"Parámetros congelados: {total_params - trainable_params:,}")

model.to(device)

# ==================== VISUALIZACIÓN DE ARQUITECTURA ====================
print("\n" + "="*80)
print("ARQUITECTURA DEL MODELO - DistilBERT + Capas Personalizadas")
print("="*80)
print(model)
print("-"*80)
print(f"\n{'RESUMEN':^80}")
print("-"*80)
print(f"Total de parámetros:      {total_params:>20,}")
print(f"Parámetros entrenables:   {trainable_params:>20,}")
print(f"Parámetros congelados:    {total_params - trainable_params:>20,}")
print(f"Porcentaje entrenable:    {100 * trainable_params / total_params:>19.2f}%")
print("="*80 + "\n")
# ========================================================================



Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Total parámetros: 66,758,147
Parámetros entrenables: 395,267
Parámetros congelados: 66,362,880

ARQUITECTURA DEL MODELO - DistilBERT + Capas Personalizadas
DistilBertWithCustomLayers(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
        

In [41]:
# 8. Configurar optimizador y scheduler
optimizer = AdamW(model.parameters(), lr=5e-5)
num_epochs = 20
total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# 9. Función de entrenamiento (MODIFICADA)
def train_epoch(model, data_loader, optimizer, scheduler, device):
    model.train()
    losses = []
    correct_predictions = 0
    loss_fn = torch.nn.NLLLoss()  # ⬅️ CAMBIO: NLLLoss para usar con softmax

    for batch in tqdm(data_loader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Aplicar log antes de calcular la pérdida
        log_probs = torch.log(logits + 1e-10)  # Añadir epsilon para estabilidad numérica
        loss = loss_fn(log_probs, labels)

        # Calcular accuracy
        _, preds = torch.max(logits, dim=1)
        correct_predictions += torch.sum(preds == labels)

        losses.append(loss.item())

        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    return correct_predictions.double() / len(data_loader.dataset), np.mean(losses)


# 10. Función de evaluación (MODIFICADA)
def eval_model(model, data_loader, device):
    model.eval()
    losses = []
    correct_predictions = 0
    loss_fn = torch.nn.NLLLoss()  # ⬅️ CAMBIO: NLLLoss para usar con softmax

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            # Aplicar log antes de calcular la pérdida
            log_probs = torch.log(logits + 1e-10)  # Añadir epsilon para estabilidad numérica
            loss = loss_fn(log_probs, labels)

            _, preds = torch.max(logits, dim=1)
            correct_predictions += torch.sum(preds == labels)
            losses.append(loss.item())

    return correct_predictions.double() / len(data_loader.dataset), np.mean(losses)

# 11. Entrenamiento
history = {'train_acc': [], 'train_loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    print('-' * 50)

    train_acc, train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    print(f'Train loss: {train_loss:.4f}, Train accuracy: {train_acc:.4f}')

    val_acc, val_loss = eval_model(model, val_loader, device)
    print(f'Val loss: {val_loss:.4f}, Val accuracy: {val_acc:.4f}')

    history['train_acc'].append(train_acc)
    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc)
    history['val_loss'].append(val_loss)


Epoch 1/20
--------------------------------------------------


Training: 100%|██████████| 124/124 [00:11<00:00, 10.34it/s]


Train loss: 1.0080, Train accuracy: 0.4932


Evaluating: 100%|██████████| 16/16 [00:03<00:00,  5.30it/s]


Val loss: 0.9674, Val accuracy: 0.5066

Epoch 2/20
--------------------------------------------------


...

Val loss: 0.7510, Val accuracy: 0.6593

Epoch 20/20
--------------------------------------------------


Training: 100%|██████████| 124/124 [00:12<00:00, 10.16it/s]


Train loss: 0.7545, Train accuracy: 0.6595


Evaluating: 100%|██████████| 16/16 [00:03<00:00,  5.21it/s]

Val loss: 0.7505, Val accuracy: 0.6613


**Punto 2 - Nuevas métricas**

Nuevas métricas obtenidas con el modelo re-entrenado!

In [42]:
# 12. Evaluación final en training
test_acc, test_loss = eval_model(model, train_loader, device)
print(f'\nTrain loss: {test_loss:.4f}, Train accuracy: {test_acc:.4f}')

print('\n' * 2)

# 12. Evaluación final en test
test_acc, test_loss = eval_model(model, test_loader, device)
print(f'\nTest loss: {test_loss:.4f}, Test accuracy: {test_acc:.4f}')

Evaluating: 100%|██████████| 124/124 [00:11<00:00, 10.40it/s]



Train loss: 0.7412, Train accuracy: 0.6673





Evaluating: 100%|██████████| 20/20 [00:03<00:00,  5.20it/s]


Test loss: 0.7733, Test accuracy: 0.6613


**Punto 3**

A continuación, el código para realizar fine-tuning al modelo base del laboratorio

***Reconfiguración del modelo base***

se realizan los siguientes cambios para proceder con ajuste fino:



In [57]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm
import numpy as np

# 1. Verificar si hay GPU disponible
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando dispositivo: {device}')

# 2. Cargar el tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# 3. Tokenizar y aplicar padding a los dataframes
X_train = tokenizer(
    train['Sentence'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'  # PyTorch tensors
)

X_val = tokenizer(
    val['Sentence'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

X_test = tokenizer(
    test['Sentence'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

# 4. Crear clase Dataset personalizada
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item

# 5. Crear datasets
train_dataset = SentimentDataset(X_train, train['Sentiment'])
val_dataset = SentimentDataset(X_val, val['Sentiment'])
test_dataset = SentimentDataset(X_test, test['Sentiment'])

# 6. Crear DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 7. Crear modelo personalizado con capas adicionales
class DistilBertWithCustomLayers(torch.nn.Module):
    def __init__(self, num_labels=3):
        super(DistilBertWithCustomLayers, self).__init__()

        # Cargar DistilBERT base
        self.distilbert = DistilBertForSequenceClassification.from_pretrained(
            'distilbert-base-uncased',
            num_labels=num_labels
        ).distilbert

        # Capas personalizadas
        self.dense_hidden = torch.nn.Linear(768, 512)
        self.sigmoid = torch.nn.Sigmoid()
        self.dropout = torch.nn.Dropout(0.2)
        self.classifier = torch.nn.Linear(512, num_labels)
        self.softmax = torch.nn.Softmax(dim=1)

    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        hidden_state = outputs.last_hidden_state[:, 0, :]
        x = self.dense_hidden(hidden_state)
        x = self.sigmoid(x)
        x = self.dropout(x)
        x = self.classifier(x)
        logits = self.softmax(x)
        return logits

# Instanciar el modelo
model = DistilBertWithCustomLayers(num_labels=3)

# ============ DESCONGELAR DISTILBERT PARA FINE-TUNING ============
for param in model.distilbert.parameters():
    param.requires_grad = True  # ⬅️ Entrenar todo el modelo
print("🔥 Fine-tuning habilitado: DistilBERT se entrenará")
# ==================================================================

# Contar parámetros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parámetros: {total_params:,}")
print(f"Parámetros entrenables: {trainable_params:,}")
print(f"Parámetros congelados: {total_params - trainable_params:,}")
print(f"Porcentaje entrenable: {100 * trainable_params / total_params:.2f}%")

model.to(device)

# 8. Configurar optimizador con learning rates diferenciados
optimizer = AdamW([
    {'params': model.distilbert.parameters(), 'lr': 1e-5},      # LR bajo para pre-entrenado
    {'params': model.dense_hidden.parameters(), 'lr': 1e-4},    # LR alto para capas nuevas
    {'params': model.classifier.parameters(), 'lr': 1e-4}
])

num_epochs = 10  # Fine-tuning: menos épocas
total_steps = len(train_loader) * num_epochs

# Warmup: 10% de los pasos
num_warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=total_steps
)

print(f"\n📊 Configuración de entrenamiento:")
print(f"  Épocas: {num_epochs}")
print(f"  Pasos totales: {total_steps}")
print(f"  Warmup steps: {num_warmup_steps}")
print(f"  LR DistilBERT: 2e-5")
print(f"  LR Capas personalizadas: 5e-4")

Usando dispositivo: cuda


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🔥 Fine-tuning habilitado: DistilBERT se entrenará

Total parámetros: 66,758,147
Parámetros entrenables: 66,758,147
Parámetros congelados: 0
Porcentaje entrenable: 100.00%

📊 Configuración de entrenamiento:
  Épocas: 10
  Pasos totales: 1240
  Warmup steps: 124
  LR DistilBERT: 2e-5
  LR Capas personalizadas: 5e-4


**Punto 3 - Nuevo entrenamiento**

A continuación, el entrenamiento con el modelo reconfigurado para ajuste fino

In [58]:
# 9. Función de entrenamiento (MODIFICADA)
def train_epoch(model, data_loader, optimizer, scheduler, device):
    model.train()
    losses = []
    correct_predictions = 0
    loss_fn = torch.nn.NLLLoss()  # ⬅️ CAMBIO: NLLLoss para usar con softmax

    for batch in tqdm(data_loader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Aplicar log antes de calcular la pérdida
        log_probs = torch.log(logits + 1e-10)  # Añadir epsilon para estabilidad numérica
        loss = loss_fn(log_probs, labels)

        # Calcular accuracy
        _, preds = torch.max(logits, dim=1)
        correct_predictions += torch.sum(preds == labels)

        losses.append(loss.item())

        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    return correct_predictions.double() / len(data_loader.dataset), np.mean(losses)


# 10. Función de evaluación (MODIFICADA)
def eval_model(model, data_loader, device):
    model.eval()
    losses = []
    correct_predictions = 0
    loss_fn = torch.nn.NLLLoss()  # ⬅️ CAMBIO: NLLLoss para usar con softmax

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            # Aplicar log antes de calcular la pérdida
            log_probs = torch.log(logits + 1e-10)  # Añadir epsilon para estabilidad numérica
            loss = loss_fn(log_probs, labels)

            _, preds = torch.max(logits, dim=1)
            correct_predictions += torch.sum(preds == labels)
            losses.append(loss.item())

    return correct_predictions.double() / len(data_loader.dataset), np.mean(losses)

# 11. Entrenamiento
history = {'train_acc': [], 'train_loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    print('-' * 50)

    train_acc, train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    print(f'Train loss: {train_loss:.4f}, Train accuracy: {train_acc:.4f}')

    val_acc, val_loss = eval_model(model, val_loader, device)
    print(f'Val loss: {val_loss:.4f}, Val accuracy: {val_acc:.4f}')

    history['train_acc'].append(train_acc)
    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc)
    history['val_loss'].append(val_loss)


Epoch 1/10
--------------------------------------------------


Training: 100%|██████████| 124/124 [00:37<00:00,  3.33it/s]


Train loss: 0.9425, Train accuracy: 0.5513


Evaluating: 100%|██████████| 16/16 [00:03<00:00,  5.13it/s]


Val loss: 0.6630, Val accuracy: 0.7230

Epoch 2/10
--------------------------------------------------


...

Val loss: 0.5379, Val accuracy: 0.7796

Epoch 10/10
--------------------------------------------------


Training: 100%|██████████| 124/124 [00:36<00:00,  3.37it/s]


Train loss: 0.1456, Train accuracy: 0.9280


Evaluating: 100%|██████████| 16/16 [00:03<00:00,  5.21it/s]

Val loss: 0.5361, Val accuracy: 0.7836


**Punto 3 - Nueva evaluación**

Con el modelo reentrenado con ajuste fino se realiza una nueva evaluación y a continuación las nuevas métricas obtenidas:

In [59]:
# 12. Evaluación final en training
test_acc, test_loss = eval_model(model, train_loader, device)
print(f'\nTrain loss: {test_loss:.4f}, Train accuracy: {test_acc:.4f}')

print('\n' * 2)

# 12. Evaluación final en test
test_acc, test_loss = eval_model(model, test_loader, device)
print(f'\nTest loss: {test_loss:.4f}, Test accuracy: {test_acc:.4f}')

Evaluating: 100%|██████████| 124/124 [00:11<00:00, 10.35it/s]



Train loss: 0.1285, Train accuracy: 0.9353





Evaluating: 100%|██████████| 20/20 [00:03<00:00,  5.21it/s]


Test loss: 0.5826, Test accuracy: 0.7769
